# 🚀 Урок 20 · Gradio и деплой: приложение за 10 минут (материалы преподавателя)

Превращаем **настоящую модель** (наш классификатор пингвинов) в веб-приложение и выкладываем его в интернет навсегда через Hugging Face Spaces. Ссылку можно отправить родителям прямо с урока.

> План: 1) обучаем и сохраняем модель → 2) быстрый Gradio (временная ссылка) → 3) готовим файлы для Spaces → 4) деплой → 5) постоянная ссылка.

Почему маленькая sklearn-модель, а не тяжёлый transformers: на free Spaces она собирается за 2–3 минуты (нужен только `gradio` + `scikit-learn`), и это настоящая ML-модель курса, а не хардкод.

## Разница: временная ссылка vs постоянная

- **Gradio `share=True`** (урок 16) — ссылка живёт, пока открыт ноутбук (до 72 ч). Быстро, но временно.
- **Hugging Face Spaces** — приложение живёт в интернете ВСЕГДА, ссылка постоянная. Это настоящий деплой.

Сегодня освоим второе.

## Шаг 1 · Обучаем и сохраняем модель
Берём знакомый датасет пингвинов и обучаем классификатор вида по 4 измерениям. `joblib.dump` сохраняет обученную модель в файл — его потом загрузим в приложении.

In [ ]:
import seaborn as sns, joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

df = sns.load_dataset('penguins').dropna()
features = ['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']
X = df[features].values          # 4 числа-измерения
y = df['species'].values         # вид пингвина
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

model = Pipeline([('scaler', StandardScaler()),
                  ('clf', RandomForestClassifier(random_state=42))])
model.fit(Xtr, ytr)
print('Точность на тесте:', round(model.score(Xte, yte), 3))

joblib.dump(model, 'penguin_model.joblib')   # сохраняем модель в файл
print('Модель сохранена. Классы:', list(model.classes_))

## Шаг 2 · Быстрое приложение с временной ссылкой (разминка)
Знакомый способ из урока 16: оборачиваем модель в Gradio и получаем временную ссылку `*.gradio.live`.

In [ ]:
!pip install gradio -q
import gradio as gr, joblib

model = joblib.load('penguin_model.joblib')

def predict(bill_length, bill_depth, flipper_length, body_mass):
    proba = model.predict_proba([[bill_length, bill_depth, flipper_length, body_mass]])[0]
    return {cls: float(p) for cls, p in zip(model.classes_, proba)}

demo = gr.Interface(
    fn=predict,
    inputs=[gr.Number(label='Длина клюва, мм', value=45),
            gr.Number(label='Глубина клюва, мм', value=17),
            gr.Number(label='Длина крыла, мм', value=200),
            gr.Number(label='Масса, г', value=4000)],
    outputs=gr.Label(label='Вид пингвина'),
    title='Определитель пингвина',
    description='Введи измерения — модель определит вид')

demo.launch(share=True)   # временная ссылка gradio.live

**❓ Вопрос 1.** Открой временную ссылку на телефоне, проверь на своих числах. Что случится с этой ссылкой, если закрыть ноутбук?

## Шаг 3 · Готовим файлы для постоянного деплоя
Для Spaces нужны **два файла, которые мы пишем**, плюс **файл модели**:
- `app.py` — код приложения (тот же Gradio, что выше, но `launch()` без `share=True`),
- `requirements.txt` — список библиотек,
- `penguin_model.joblib` — уже сохранён на шаге 1.

`%%writefile` сохраняет содержимое ячейки в файл.

In [ ]:
%%writefile app.py
import gradio as gr, joblib

model = joblib.load('penguin_model.joblib')   # модель лежит рядом на Spaces

def predict(bill_length, bill_depth, flipper_length, body_mass):
    proba = model.predict_proba([[bill_length, bill_depth, flipper_length, body_mass]])[0]
    return {cls: float(p) for cls, p in zip(model.classes_, proba)}

demo = gr.Interface(
    fn=predict,
    inputs=[gr.Number(label='Длина клюва, мм', value=45),
            gr.Number(label='Глубина клюва, мм', value=17),
            gr.Number(label='Длина крыла, мм', value=200),
            gr.Number(label='Масса, г', value=4000)],
    outputs=gr.Label(label='Вид пингвина'),
    title='Определитель пингвина')

demo.launch()   # без share=True — на Spaces адрес даётся сам

In [ ]:
%%writefile requirements.txt
gradio
scikit-learn
joblib

In [ ]:
# проверим, что всё на месте
import os
for f in ['app.py','requirements.txt','penguin_model.joblib']:
    print(f, '—', 'есть' if os.path.exists(f) else 'НЕТ')

## Шаг 4 · Деплой на Hugging Face Spaces (пошагово)
Способ через сайт — самый надёжный, без токенов и терминала.

1. Зайди на **huggingface.co** и создай бесплатный аккаунт (если ещё нет).
2. Аватар → **New Space**.
3. Заполни: **Space name** (например `penguin-classifier`), **SDK → Gradio**, **Public**.
4. Нажми **Create Space**.
5. Вкладка **Files** → **Add file** → **Upload files**.
6. Загрузи **ТРИ файла**: `app.py`, `requirements.txt`, `penguin_model.joblib`.
7. **Commit changes**.
8. Вкладка **App** — подожди 2–3 минуты (Building), приложение запустится.

**Готово!** Постоянная ссылка: `huggingface.co/spaces/ТВОЙ_ЛОГИН/penguin-classifier`

> Совет: если во вкладке App ошибка — открой **Logs**, там видно, какой библиотеки не хватило (или что забыл загрузить файл модели). Добавь/загрузи и сделай commit заново.

## Шаг 5 · Скачиваем файлы из Colab для загрузки
Запусти — три файла скачаются на компьютер, потом загрузишь их на Spaces (шаг 4, пункт 6).

In [ ]:
from google.colab import files
files.download('app.py')
files.download('requirements.txt')
files.download('penguin_model.joblib')

---
## 🎯 Задания

### 🟢 Базовый
Задеплой определитель пингвина на Spaces. Получи постоянную ссылку, открой на телефоне, отправь однокласснику.

### 🟡 Продвинутый
Добавь в `app.py` примеры (`gr.Examples`) с реальными измерениями трёх видов и красивое описание.

### ⭐ Со звёздочкой
Замени модель на свой классификатор из другого урока (например фото из урока 16 — вход `gr.Image()`). Это прямая репетиция финального проекта и Demo Day!

## Мини-итог

- `share=True` даёт ... ссылку, а Spaces даёт ... ссылку.
- Для деплоя на Spaces нужны файлы: ..., ... и ...
- Если приложение не запустилось, смотреть надо во вкладку ...

> У тебя теперь работающее приложение в интернете с постоянной ссылкой — настоящий продукт для Demo Day.